Load and Combine All Data

In [1]:
import pandas as pd
import glob
import os

# Define your folders
flow_level_folder = r"C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\models\model_main\data\flow_level\flow_data_test"
packet_level_folder = r"C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\models\model_main\data\packet_level"

# Collect all parquet files in each folder
flow_level_files = glob.glob(os.path.join(flow_level_folder, "*.parquet"))
packet_level_files = glob.glob(os.path.join(packet_level_folder, "*.parquet"))

print("Flow-level files:", flow_level_files)
print("Packet-level files:", packet_level_files)

# Load all flow-level data (parquet)
flow_dfs = []
for attack_file in flow_level_files:  # Your 12 parquet files
    df = pd.read_parquet(attack_file)
    flow_dfs.append(df)
combined_flows = pd.concat(flow_dfs, ignore_index=True)

# Load all packet-level data (parquet)
packet_dfs = []
for attack_file in packet_level_files:  # Your parquet attack type files
    df = pd.read_parquet(attack_file)
    packet_dfs.append(df)
combined_packets = pd.concat(packet_dfs, ignore_index=True)

combined_flows.to_parquet(r"C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\models\model_main\data\combined_data\combined_flows.parquet", index=False)
combined_packets.to_parquet(r"C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\models\model_main\data\combined_data\combined_packets.parquet", index=False)

Flow-level files: ['C:\\Users\\AGFirass\\Documents\\GitHub\\Transformer-Based-DDoS-Detection\\models\\model_main\\data\\flow_level\\flow_data_test\\DrDoS_DNS_5k_samples.parquet', 'C:\\Users\\AGFirass\\Documents\\GitHub\\Transformer-Based-DDoS-Detection\\models\\model_main\\data\\flow_level\\flow_data_test\\DrDoS_LDAP_5k_samples.parquet', 'C:\\Users\\AGFirass\\Documents\\GitHub\\Transformer-Based-DDoS-Detection\\models\\model_main\\data\\flow_level\\flow_data_test\\DrDoS_MSSQL_5k_samples.parquet']
Packet-level files: ['C:\\Users\\AGFirass\\Documents\\GitHub\\Transformer-Based-DDoS-Detection\\models\\model_main\\data\\packet_level\\DrDos_Dns_100_packets_per_flow.parquet', 'C:\\Users\\AGFirass\\Documents\\GitHub\\Transformer-Based-DDoS-Detection\\models\\model_main\\data\\packet_level\\DrDoS_LDAP_100_packets_per_flow.parquet', 'C:\\Users\\AGFirass\\Documents\\GitHub\\Transformer-Based-DDoS-Detection\\models\\model_main\\data\\packet_level\\DrDoS_MSSQL_100_packets_per_flow.parquet']


Create Flow Dictionary

In [9]:
# Remove leading/trailing spaces from all column names
combined_flows.columns = combined_flows.columns.str.strip()
combined_packets.columns = combined_packets.columns.str.strip()

In [10]:
# Group packets by flow_id (they're already sorted by timestamp - perfect!)
flows_dict = {}

for flow_id in combined_flows['Flow ID'].unique():
    # Flow-level features
    flow_row = combined_flows[combined_flows['Flow ID'] == flow_id].iloc[0]

    # Packet-level features for this flow
    flow_packets = combined_packets[combined_packets['Flow ID'] == flow_id]

    flows_dict[flow_id] = {
        'flow_features': flow_row.drop(['Flow ID', 'Label']).values,  # 50 features
        'packets': flow_packets.drop(['Flow ID', 'label'], axis=1).values,  # Nx25 features
        'label': flow_row['Label'],
        'attack_type': flow_row['Label']
    }


In [8]:
print("Flows :: ", combined_flows.columns)
print("Packets :: ", combined_packets.columns)

Flows ::  Index(['Unnamed: 0', 'Flow ID', ' Source IP', ' Source Port',
       ' Destination IP', ' Destination Port', ' Protocol', ' Timestamp',
       ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets',
       'Total Length of Fwd Packets', ' Total Length of Bwd Packets',
       ' Fwd Packet Length Max', ' Fwd Packet Length Min',
       ' Fwd Packet Length Mean', ' Fwd Packet Length Std',
       'Bwd Packet Length Max', ' Bwd Packet Length Min',
       ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s',
       ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max',
       ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std',
       ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean',
       ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags',
       ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags',
       ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s',
       ' Bwd Packets/s', ' Min 

Create Multi-Flow Training Samples

In [15]:
import random


def create_multi_flow_samples(flows_dict, flows_per_sample=6):
    training_samples = []
    flow_ids = list(flows_dict.keys())

    # Shuffle to ensure random combinations of attack types
    random.shuffle(flow_ids)

    for i in range(0, len(flow_ids), flows_per_sample):
        sample_flow_ids = flow_ids[i:i+flows_per_sample]

        if len(sample_flow_ids) == flows_per_sample:  # Ensure consistent sample size
            sample_flows = [flows_dict[fid] for fid in sample_flow_ids]

            # Sample label strategy: multi-hot encoding for all attack types present
            attack_types = [flow['attack_type'] for flow in sample_flows]
            sample_label = attack_types  # List of all attacks in this sample

            training_samples.append({
                'flows': sample_flows,
                'sample_label': sample_label,
                'sample_id': len(training_samples)
            })

    return training_samples

training_samples = create_multi_flow_samples(flows_dict, flows_per_sample=6)
print("Training samples :: ", training_samples[:1])

Training samples ::  [{'flows': [{'flow_features': array([np.int64(2065), '172.16.0.5', np.int64(38057), '192.168.50.1',
       np.int64(20987), np.int64(17), '2018-12-01 10:54:03.295144',
       np.int64(3), np.int64(2), np.int64(0), np.float64(2944.0),
       np.float64(0.0), np.float64(1472.0), np.float64(1472.0),
       np.float64(1472.0), np.float64(0.0), np.float64(0.0),
       np.float64(0.0), np.float64(0.0), np.float64(0.0),
       np.float64(981333333.3333333), np.float64(666666.6666666666),
       np.float64(3.0), np.float64(0.0), np.float64(3.0), np.float64(3.0),
       np.float64(3.0), np.float64(3.0), np.float64(0.0), np.float64(3.0),
       np.float64(3.0), np.float64(0.0), np.float64(0.0), np.float64(0.0),
       np.float64(0.0), np.float64(0.0), np.int64(0), np.int64(0),
       np.int64(0), np.int64(0), np.int64(64), np.int64(0),
       np.float64(666666.6666666665), np.float64(0.0), np.float64(1472.0),
       np.float64(1472.0), np.float64(1472.0), np.float64(0.0),
  

In [16]:
import numpy as np

MAX_PACKETS_PER_FLOW = 100  # Based on your data

def pad_or_truncate_packets(packets, max_length=MAX_PACKETS_PER_FLOW):
    if len(packets) > max_length:
        return packets[:max_length]  # Truncate
    else:
        # Pad with zeros
        padding = np.zeros((max_length - len(packets), packets.shape[1]))
        return np.vstack([packets, padding])

# Apply to all flows
for flow_id in flows_dict:
    flows_dict[flow_id]['packets'] = pad_or_truncate_packets(
        flows_dict[flow_id]['packets']
    )

In [23]:
print(len(flows_dict))

14606


In [24]:
flow_id = list(flows_dict.keys())[0]
print(flows_dict[flow_id])

{'flow_features': array([np.int64(425), '172.16.0.5', np.int64(634), '192.168.50.1',
       np.int64(60495), np.int64(17), '2018-12-01 10:51:39.813448',
       np.int64(28415), np.int64(97), np.int64(0), np.float64(42680.0),
       np.float64(0.0), np.float64(440.0), np.float64(440.0),
       np.float64(440.0), np.float64(0.0), np.float64(0.0),
       np.float64(0.0), np.float64(0.0), np.float64(0.0),
       np.float64(1502023.5790955485), np.float64(3413.689952489882),
       np.float64(295.98958333333337), np.float64(500.95930068517794),
       np.float64(3596.0), np.float64(1.0), np.float64(28415.0),
       np.float64(295.98958333333337), np.float64(500.95930068517794),
       np.float64(3596.0), np.float64(1.0), np.float64(0.0),
       np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0),
       np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(-97),
       np.int64(0), np.float64(3413.689952489882), np.float64(0.0),
       np.float64(440.0), np.float64(440

In [36]:
# Pick a flow
flow_id = list(flows_dict.keys())[0]
flow = flows_dict[flow_id]

# Top-level keys
print(flow.keys())

# Shape and type of flow features
print("flow_features:", type(flow['flow_features']), flow['flow_features'].shape)

# Shape and type of packets
print("packets:", type(flow['packets']), flow['packets'].shape)

# Sample first 3 packets
print("Sample packets:\n", flow['packets'][:3])

# Label and attack type
print("label:", flow['label'])
print("attack_type:", flow['attack_type'])

dict_keys(['flow_features', 'packets', 'label', 'attack_type'])
flow_features: <class 'numpy.ndarray'> (86,)
packets: <class 'numpy.ndarray'> (100, 24)
Sample packets:
 [[1543674991.466844 '172.16.0.5' '192.168.50.1' 634.0 60495.0 17 482 47
  440 34451 2 0 192 nan nan nan nan nan nan nan 1 0.0 3.901293571653097
  'forward']
 [1543674991.466845 '172.16.0.5' '192.168.50.1' 634.0 60495.0 17 482 47
  440 34451 2 0 192 nan nan nan nan nan nan nan 2 9.5367431640625e-07
  3.901293571653097 'forward']
 [1543674991.466904 '172.16.0.5' '192.168.50.1' 634.0 60495.0 17 482 47
  440 34452 2 0 192 nan nan nan nan nan nan nan 3 5.888938903808594e-05
  4.063520836895287 'forward']]
label: DrDoS_DNS
attack_type: DrDoS_DNS


In [26]:
import pickle

with open(r'C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\models\model_main\data\final_data\flows_dict.pkl', 'wb') as f:
    pickle.dump(flows_dict, f)

In [27]:
for flow_id, flow in flows_dict.items():
    flow_feat_shape = flow['flow_features'].shape
    packets_shape = flow['packets'].shape
    label = flow['label']
    attack_type = flow['attack_type']

    print(f"Flow ID: {flow_id}")
    print(f"  flow_features shape: {flow_feat_shape}")
    print(f"  packets shape: {packets_shape}")
    print(f"  label: {label}, attack_type: {attack_type}\n")


Flow ID: 172.16.0.5_192.168.50.1_634_60495_17
  flow_features shape: (86,)
  packets shape: (100, 24)
  label: DrDoS_DNS, attack_type: DrDoS_DNS

Flow ID: 172.16.0.5_192.168.50.1_634_46391_17
  flow_features shape: (86,)
  packets shape: (100, 24)
  label: DrDoS_DNS, attack_type: DrDoS_DNS

Flow ID: 172.16.0.5_192.168.50.1_634_11894_17
  flow_features shape: (86,)
  packets shape: (100, 24)
  label: DrDoS_DNS, attack_type: DrDoS_DNS

Flow ID: 172.16.0.5_192.168.50.1_634_27878_17
  flow_features shape: (86,)
  packets shape: (100, 24)
  label: DrDoS_DNS, attack_type: DrDoS_DNS

Flow ID: 172.16.0.5_192.168.50.1_634_47149_17
  flow_features shape: (86,)
  packets shape: (100, 24)
  label: DrDoS_DNS, attack_type: DrDoS_DNS

Flow ID: 172.16.0.5_192.168.50.1_634_22713_17
  flow_features shape: (86,)
  packets shape: (100, 24)
  label: DrDoS_DNS, attack_type: DrDoS_DNS

Flow ID: 172.16.0.5_192.168.50.1_634_49912_17
  flow_features shape: (86,)
  packets shape: (100, 24)
  label: DrDoS_DNS, at

In [28]:
flow_feat_shapes = set()
packets_shapes = set()

for flow in flows_dict.values():
    flow_feat_shapes.add(flow['flow_features'].shape)
    packets_shapes.add(flow['packets'].shape)

print("Unique flow_features shapes:", flow_feat_shapes)
print("Unique packets shapes:", packets_shapes)

Unique flow_features shapes: {(86,)}
Unique packets shapes: {(100, 24)}


In [29]:
for flow_id, flow in flows_dict.items():
    if flow['flow_features'].size == 0 or flow['packets'].size == 0:
        print(f"Empty flow detected: {flow_id}")


In [32]:
import numpy as np

for flow_id, flow in flows_dict.items():
    packets = flow['packets']

    # Check for rows that are all zeros or NaNs
    empty_rows = np.all((packets == 0), axis=1)
    num_empty = np.sum(empty_rows)

    if num_empty > 0:
        print(f"Flow {flow_id} has {num_empty} empty/invalid packets out of {packets.shape[0]}")


Flow 172.16.0.5_192.168.50.1_48621_22_6 has 66 empty/invalid packets out of 100
Flow 192.168.50.8_125.56.201.115_59099_80_6 has 52 empty/invalid packets out of 100
Flow 192.168.50.8_54.218.239.186_59102_443_6 has 72 empty/invalid packets out of 100
Flow 192.168.50.8_23.15.4.11_59155_80_6 has 66 empty/invalid packets out of 100
Flow 172.217.0.110_192.168.50.8_80_59131_6 has 64 empty/invalid packets out of 100
Flow 172.217.0.110_192.168.50.8_80_59132_6 has 64 empty/invalid packets out of 100
Flow 192.168.50.8_104.36.115.113_59149_443_6 has 54 empty/invalid packets out of 100
Flow 192.168.50.8_72.21.91.29_59162_80_6 has 64 empty/invalid packets out of 100
Flow 172.217.0.110_192.168.50.8_80_59134_6 has 64 empty/invalid packets out of 100
Flow 172.217.0.110_192.168.50.8_80_59133_6 has 60 empty/invalid packets out of 100
Flow 192.168.50.8_54.210.144.213_59147_443_6 has 76 empty/invalid packets out of 100
Flow 192.168.50.8_72.21.91.29_59104_80_6 has 42 empty/invalid packets out of 100
Flow 19

In [37]:
import pickle

file_path = r"C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\models\model_main\data\final_data\flows_dict.pkl"

with open(file_path, "rb") as f:
    flows_dict = pickle.load(f)

for flow_id in flows_dict:
    if 'attack_type' in flows_dict[flow_id]:
        del flows_dict[flow_id]['attack_type']

with open(file_path, "wb") as f:
    pickle.dump(flows_dict, f)

print("attack_type removed and file overwritten successfully!")

attack_type removed and file overwritten successfully!


In [40]:
import pickle

with open(r'C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\models\model_main\data\final_data\flows_dict.pkl', 'wb') as f:
    pickle.dump(flows_dict, f)

for flow_id, flow in flows_dict.items():
    flow_feat_shape = flow['flow_features'].shape
    packets_shape = flow['packets'].shape
    label = flow['label']

    print(f"Flow ID: {flow_id}")
    print(f"  flow_features shape: {flow_feat_shape}")
    print(f"  packets shape: {packets_shape}")
    print(f"  label: {label}\n")

Flow ID: 172.16.0.5_192.168.50.1_634_60495_17
  flow_features shape: (86,)
  packets shape: (100, 24)
  label: DrDoS_DNS

Flow ID: 172.16.0.5_192.168.50.1_634_46391_17
  flow_features shape: (86,)
  packets shape: (100, 24)
  label: DrDoS_DNS

Flow ID: 172.16.0.5_192.168.50.1_634_11894_17
  flow_features shape: (86,)
  packets shape: (100, 24)
  label: DrDoS_DNS

Flow ID: 172.16.0.5_192.168.50.1_634_27878_17
  flow_features shape: (86,)
  packets shape: (100, 24)
  label: DrDoS_DNS

Flow ID: 172.16.0.5_192.168.50.1_634_47149_17
  flow_features shape: (86,)
  packets shape: (100, 24)
  label: DrDoS_DNS

Flow ID: 172.16.0.5_192.168.50.1_634_22713_17
  flow_features shape: (86,)
  packets shape: (100, 24)
  label: DrDoS_DNS

Flow ID: 172.16.0.5_192.168.50.1_634_49912_17
  flow_features shape: (86,)
  packets shape: (100, 24)
  label: DrDoS_DNS

Flow ID: 172.16.0.5_192.168.50.1_634_56681_17
  flow_features shape: (86,)
  packets shape: (100, 24)
  label: DrDoS_DNS

Flow ID: 172.16.0.5_192.